# 04. Layer 3 — Coordination（對等型多 Agent 協作）

**情境**：使用者給一個技術主題（例如「RAG 系統入門」），系統產出一份結構化文件。團隊成員：

- 🧭 **Coordinator** — 入口，決定第一步該找誰
- 📋 **Outline Agent** — 規劃文件大綱
- ✍️ **Writer Agent** — 根據大綱寫內容
- 🔍 **Reviewer Agent** — 校對 / 給回饋

**Coordination 跟 Orchestration 的關鍵差別**：流程**不是寫死的**。Coordinator 只決定起點，後面誰交給誰、要不要回頭找 reviewer 改稿，**全部由 LLM 動態決定**。

ADK 對應的機制：`LlmAgent` 的 `sub_agents` + `transfer_to_agent` 自動工具 + `disallow_transfer_to_peers=False`。

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import get_model, final_text

## 1. 三個專家 Agent

每個 sub-agent 都要：
1. **設 `description`**：這是 coordinator 用來判斷「該不該轉給這個人」的說明
2. **`disallow_transfer_to_peers=False`**：允許橫向轉給其他 peer（這就是 Coordination 的精髓）
3. **`disallow_transfer_to_parent=False`**：完成後可以把控制權還給 coordinator

> ⚠️ 這兩個屬性的預設值是 `True`（不允許轉），所以**沒設等於沒有 Coordination**，新手很容易踩坑。

In [2]:
from google.adk.agents import LlmAgent

# ⚠️ 設計重點：instruction 要**直接命令模型呼叫工具**，不能只「描述」工具。
# gpt-oss / 開源模型常見坑：模型會把 tool call 序列化成 JSON 文字
# (e.g. 印出 `{"agent_name":"writer_agent"}`) 而不是真的觸發 function call。
# 解法：明確說「呼叫 transfer_to_agent(agent_name='X')」，並補一句「不要序列化成文字」。

outline_agent = LlmAgent(
    name="outline_agent",
    model=get_model(),
    description="規劃技術文件的大綱結構（章節 + 每章重點）。",
    instruction=(
        "你是技術文件大綱規劃師。產出 3~5 章大綱，每章標題 + 2 個 bullet。\n"
        "**完成後務必呼叫 `transfer_to_agent(agent_name='writer_agent')` 工具**。\n"
        "不要把工具呼叫寫成 JSON 文字，要真的觸發 function call。"
    ),
    disallow_transfer_to_peers=False,
    disallow_transfer_to_parent=False,
)

writer_agent = LlmAgent(
    name="writer_agent",
    model=get_model(),
    description="根據大綱撰寫技術文件正文。",
    instruction=(
        "你是技術寫手。讀對話前面的大綱，每章寫 1~2 段繁體中文內容，**保持簡潔**。\n"
        "**寫完後務必呼叫 `transfer_to_agent(agent_name='reviewer_agent')` 工具**讓 reviewer 校對。\n"
        "不要序列化成 JSON 文字。"
    ),
    disallow_transfer_to_peers=False,
    disallow_transfer_to_parent=False,
)

reviewer_agent = LlmAgent(
    name="reviewer_agent",
    model=get_model(),
    description="校對技術文件，指出明顯錯誤或結構問題。",
    instruction=(
        "你是技術文件編輯。挑出最多 2 點需要修改的地方（格式：『問題 → 建議修法』）。"
        "如果文件已經 OK，回『無重大問題』。**不要重寫整篇文章**。"
    ),
    disallow_transfer_to_peers=False,
    disallow_transfer_to_parent=False,
)

## 2. Coordinator — 決定起點

Coordinator 的 `instruction` 只負責**第一步路由**，不需要寫死後續流程。後續每個 agent 自己決定下一步轉給誰（用 `transfer_to_agent` 工具，這是 ADK 自動注入的）。

In [3]:
coordinator = LlmAgent(
    name="doc_coordinator",
    model=get_model(),
    instruction=(
        "你是技術文件團隊的 PM。當使用者要產出技術文件時，"
        "**永遠先呼叫 `transfer_to_agent(agent_name='outline_agent')` 工具**（先有大綱再有內容）。\n"
        "不要自己寫文件、不要序列化成 JSON 文字、不要回答其他問題。"
    ),
    sub_agents=[outline_agent, writer_agent, reviewer_agent],
)

print("Coordinator 的 sub_agents：")
for sa in coordinator.sub_agents:
    print(f"  - {sa.name}: {sa.description}")

Coordinator 的 sub_agents：
  - outline_agent: 規劃技術文件的大綱結構（章節 + 每章重點）。
  - writer_agent: 根據大綱撰寫技術文件正文。
  - reviewer_agent: 校對技術文件，指出明顯錯誤或結構問題。


## 3. 執行 — 觀察 Agent 之間的傳遞

下面執行時會看到一連串 event，注意每個 event 的 `author` — 那就是當下接手的 agent。`transfer_to_agent` 是 ADK 自動注入的工具，事件流裡會有對應的 function_call / function_response。

In [4]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER, SID = "layer3_coord", "sean", "doc-1"
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID)
runner = Runner(agent=coordinator, app_name=APP, session_service=session_service)

topic = "RAG（Retrieval-Augmented Generation）系統入門"
msg = types.Content(role="user", parts=[types.Part(text=f"幫我寫一份「{topic}」的技術文件")])

transfer_log = []
final_outputs = []
async for ev in runner.run_async(user_id=USER, session_id=SID, new_message=msg):
    for call in ev.get_function_calls():
        if call.name == "transfer_to_agent":
            transfer_log.append((ev.author, call.args.get("agent_name")))
            print(f"  🔀 [{ev.author}] → transfer to {call.args.get('agent_name')}")
    if ev.is_final_response():
        text = final_text(ev)
        if text:
            final_outputs.append((ev.author, text))
            print(f"\n🎯 [{ev.author}]\n{text}\n" + "─" * 50)

print(f"\n=== Transfer 軌跡（共 {len(transfer_log)} 次）===")
for src, dst in transfer_log:
    print(f"  {src} → {dst}")

print(f"\n=== 最終輸出來自：{[a for a, _ in final_outputs]} ===")

  🔀 [doc_coordinator] → transfer to outline_agent


  🔀 [outline_agent] → transfer to writer_agent



🎯 [writer_agent]
**第一章 什麼是 RAG**  
 Retrieval‑Augmented Generation（RAG）是一種將資訊檢索機制嵌入至產生式 AI 工作流的架構，主要分為 three 步驟：先在外部知識庫中搜尋相關文件，再利用檢索得到的文本進行訊息增強，最後由大型語言模型依此背景生成答案。這樣的設計使得模型即便容量有限，也能透過最新且專業的資料彈性地提供高品質回答，解決了純粹靠內建參數記憶所造成的時效性與領域覆蓋不足問題。

**第二章 關鍵組件設計**  
 向量化資料庫是 RAG 系統的基石；常見選擇包括 Pinecone、FAISS、Milvus 等，它們各有不同的索引類型與相似度衡量方式，例如 IVF、HNSW 或 PQ 壓縮，需要根據資料規模與查詢速率做權衡。另一方面，原始文件需經過切片（chunk）處理，一般以 200–500 token 作為單位，同時產出摘要或 meta 信息，以降低噪聲並提高檢索精度——這些都是提升召回質量的重要實作細節。

**第三章 檢索與提示工程**  
 混合檢索同時結合稀疏 BM25 與 dense embedding 搜尋，可兼顧詞頻重要性與語意匹配，典型做法是在兩者之間設定加權比例或交叉驗證篩選候選。Prompt 設計則聚焦於如何把檢索結果有效植入 LLM 輸入，如使用 “Conditioned Question” 模板加入檢索斷句，或採用 “Answer Refine” 流程先生成粗稿再請模型校正，使最終輸出更符合情境需求。

**第四章 系統整合與部署**  
 以 LangChain 為例，只要建立 DocumentStore → Retriever → Generator 三個鏈接，即可完成從資料載入到答复的完整流水線；Haystack 則提供額外的 Pipeline DSL 幫助快速拼裝多模型服務。在部署層面，要注意 GPU/CPU 資源配置、批次大小與快取策略，以降低 latency；同時針對敏感資料加密傳輸、存儲訪問控制等安全措施不可忽視。

**第五章 效能評估與未來拓展**  
 衡量 RAG 成功與否的指標包含 recall@k、BLEU/METEOR 類自然語言准確度，以及每次查詢的算力與金錢花費；藉由 A/B 測試可以找出最佳的檢索深度與 prompt 長度平

## 4. 你剛剛親眼看到的「失敗模式」

如果你執行剛才的 cell，可能會發現：

- ✅ Coordinator → outline_agent 的 transfer 觸發了
- ✅ outline_agent → writer_agent 的 transfer 觸發了
- ❌ writer_agent 寫完後印出 `<function name="transfer_to_agent">{...}</function>` 的**文字**，但**沒有真的呼叫** transfer 工具

這就是 Coordination 真實世界的痛點 — **LLM 把 function call 序列化成 JSON 文字**，鏈就斷了。對 gpt-oss / 開源模型尤其常見。生產環境的解法：

1. **Instruction 加強防呆**：明寫「不要序列化、要真的呼叫」（我們上面已經這樣做了，但不一定每次都成功）
2. **包一層 Safety Net**：用 `after_model_callback` 偵測 response 裡有沒有 transfer 文字，有的話手動觸發
3. **混搭 Workflow**：把高風險的 chain 用 `SequentialAgent` 寫死，只在「需要靈活判斷」的點放 Coordination
4. **換成 function-calling 訓練更扎實的模型**（Gemini、GPT-4o、Claude）

## 5. 跟 Orchestration 的對比

| 對比項 | Orchestration（03） | Coordination（04） |
|-------|-------------------|------------------|
| 流程順序 | `SequentialAgent` 寫死 | `transfer_to_agent` 動態決定 |
| 控制權 | 開發者 | LLM |
| 可預測性 | 高（同輸入同路徑） | 中（同輸入不同路徑都可能）|
| 可彈性 | 低（要加分支就要改程式） | 高（只要改 instruction）|
| 失敗模式 | 任一步失敗整條停 | LLM 可能轉錯人、轉不回來、序列化成文字（剛剛看到的）|
| 推薦場景 | 訂單流程、ETL | 創作、研究、開放式問題 |

**設計判斷**：當你發現 Orchestration 的程式裡寫了一堆 if/else 來分支，就該轉 Coordination。當你發現 Coordination 跑出來的結果常常超過你想要的範圍，就該轉 Orchestration。或是混搭 — 這也是真實系統最常見的選擇。

## 結論

你已經把 ADK 三層全部走完。下一站不是 notebook，是 **Streamlit demo** — `demo/app_orchestration.py` 跟 `demo/app_coordination.py`，把這兩個概念變成可互動、可截圖的視覺化作品。